# AF2 + class-selective DLRBC — train-only routed screen
Audit complementarity pada train, static gate, lalu satu screening seed 42. AF2 dibekukan; hanya residual kelas terpilih yang dilatih. Tidak membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, tarfile, time, zipfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-selective-dlrbc'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
os.chdir(REPO)
for key in list(sys.modules):
    if key=='coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=(
 'bundles/faruq-development-v3-grouped.tar',
 'bundles/af2-direct-from-pretrained-seed42-state.zip',
 'experiments/faruq-v3-dlrbc-fresh-v1/DLRBC_FRESH/DLRBC_FRESH_seed42/weights/best.pt',
))
ARCHIVE=require_project_artifact(PROJECT,'bundles/faruq-development-v3-grouped.tar')
AF2_STATE=require_project_artifact(PROJECT,'bundles/af2-direct-from-pretrained-seed42-state.zip')
DLRBC=require_project_artifact(PROJECT,'experiments/faruq-v3-dlrbc-fresh-v1/DLRBC_FRESH/DLRBC_FRESH_seed42/weights/best.pt')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as stream: stream.extractall('/content',filter='data')
STATE=Path('/content/af2-direct-state')
if not STATE.exists():
    with zipfile.ZipFile(AF2_STATE) as stream: stream.extractall(STATE)
BASE=STATE/'af2-direct-from-pretrained-seed42-v1'
AF2=BASE/'AF2DIRECT/AF2DIRECT_seed42/weights/best.pt'
AF2_SUMMARY=BASE/'af2_direct_seed42_summary.json'
AF2_VAL=BASE/'val_reports/AF2DIRECT_seed42_val.json'
GROUPED=DATA/'faruq_grouped_summary.json'
assert all(path.is_file() for path in (AF2,AF2_SUMMARY,AF2_VAL,DLRBC,GROUPED))
assert not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-class-selective-dlrbc-v1'
REPORTS=OUTPUT/'train_only_reports'; REPORTS.mkdir(parents=True,exist_ok=True)
print('GPU/project siap:',PROJECT)

In [ ]:
import contextlib, json, torch
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
from coffee_detector.evaluate import evaluate
from coffee_detector.analysis.af2_dlrbc_complementarity import build_af2_dlrbc_complementarity_audit
AF2_TRAIN=REPORTS/'AF2DIRECT_seed42_train.json'; DLRBC_TRAIN=REPORTS/'DLRBC_FRESH_seed42_train.json'
for checkpoint,report,name in ((AF2,AF2_TRAIN,'AF2'),(DLRBC,DLRBC_TRAIN,'DLRBC')):
    if not report.is_file():
        print('EVALUATE TRAIN-ONLY:',name,flush=True)
        with (REPORTS/f'{name}_train_eval.log').open('w') as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            evaluate(checkpoint,DATA,report,split='train',device='0')
COMPLEMENT=OUTPUT/'complementarity_train_only.json'
audit=build_af2_dlrbc_complementarity_audit(DATA,AF2_TRAIN,DLRBC_TRAIN,COMPLEMENT)
print(json.dumps({k:audit[k] for k in ('selected_class_ids','selected_class_names','gates','decision')},indent=2))
assert audit['decision']=='AUTHORIZE_AF2CSD1','STOP: tidak ada complementarity train-only yang cukup.'

In [ ]:
import yaml
from coffee_detector.af2_selective_dlrbc.audit import run_af2_selective_static_audit
cfg=yaml.safe_load((REPO/'configs/af2_selective_dlrbc/AF2CSD1.yaml').read_text())
STATIC=OUTPUT/'static_audit.json'
static=run_af2_selective_static_audit(REPO/'configs/coffee_fg/models/yolo26n-p3.yaml',AF2,cfg['afab'],audit['selected_class_ids'],STATIC,device='cpu')
print(json.dumps({'parameters':{'source':static['source_parameters'],'candidate':static['candidate_parameters'],'added':static['added_parameters']},'gates':static['gates'],'decision':static['decision']},indent=2))
assert static['decision']=='PASS','STOP: static wiring gagal.'

In [ ]:
LOG=OUTPUT/'AF2CSD1_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_selective_dlrbc',
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--af2-checkpoint',str(AF2),
 '--af2-summary',str(AF2_SUMMARY),'--af2-val-report',str(AF2_VAL),
 '--complementarity-audit',str(COMPLEMENT),'--static-audit',str(STATIC),
 '--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME AF2CSD1 | log=',LOG,flush=True)
with LOG.open('a') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
last_status=None
while process.poll() is None:
    csv_path=OUTPUT/'AF2CSD1/AF2CSD1_seed42/results.csv'
    epochs=max(0,sum(1 for _ in csv_path.open())-1) if csv_path.is_file() else 0
    status=f'AF2CSD1: {epochs}/20 epoch tercatat'
    if status!=last_status: print(status,flush=True); last_status=status
    time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'AF2CSD1 gagal: {process.returncode}')
RESULT=OUTPUT/'val_reports/AF2CSD1_seed42_result.json'
result=json.loads(RESULT.read_text()); print(json.dumps({k:result[k] for k in ('baseline','candidate','deltas','selected_class_mean_map_delta','gates','decision','next')},indent=2))